# Kumo Engine task examples with TPC-H on Databricks

Explore binary classification, regression, and link prediction with Kumo Engine on TPC-H data. The notebook trains and scores the binary task by default; the other query blocks are commented alternatives.

This walkthrough defaults to a **classic Databricks GPU cluster with dedicated access mode**. The same notebook can be adapted for Serverless GPU by switching the scratch path and enabling the commented training workaround below. It reads the built-in `samples.tpch` dataset, materializes a local graph snapshot, and retains model bundles in a Unity Catalog Volume. No external credentials are required.

Each user is responsible for checking the content of datasets and the applicable licenses and determining if suitable for the intended use


## Prepare your environment

**Runtime choices:** the default path is a classic all-purpose GPU cluster with dedicated access mode on Databricks Runtime 17.3 LTS ML (Python 3.12.3). For Serverless GPU, select a Python 3.12 AI Runtime GPU environment, use the commented `/tmp/kumoai-local` scratch setting, and enable the commented `spawn` workaround only if Optuna child processes encounter IPC authentication errors. Serverless package/runtime compatibility has not been verified end-to-end here. In both cases, GPU training runs on notebook compute; a separate SQL warehouse identified by `SQL_WAREHOUSE_ID` serves the connector's SQL work.

**Compute and storage:** choose a CUDA-capable GPU and sufficient CPU RAM/GPU memory for the graph and model. There is no verified universal GPU size or exact disk requirement for this workload. Reserve at least four times the materialized graph-index size in free local scratch, plus room for retained runs. Use `/local_disk0/tmp/kumoai-local` for ephemeral scratch on classic dedicated compute, or the commented `/tmp/kumoai-local` path on Serverless GPU; `/local_disk0` is not writable on serverless or standard access mode. `/tmp` is temporary and should not hold durable or sensitive exports. Use a pre-created Unity Catalog Volume under `/Volumes/<catalog>/<schema>/<volume>` for durable model bundles.

**Permissions and access:** attach/use the chosen GPU compute; install `kumo-engine[databricks]` from PyPI; use the SQL warehouse (`CAN USE`); read `samples.tpch` (`USE CATALOG`, `USE SCHEMA`, `SELECT`); create/replace views in the destination catalog/schema (`USE CATALOG`, `USE SCHEMA`, `CREATE TABLE`, plus `CREATE SCHEMA` on the catalog if this notebook creates that schema); and read/write the configured Volume (`READ VOLUME`, `WRITE VOLUME`). Enabling prediction output also requires permission to create or overwrite its dedicated table. Existing views must be owned or manageable by the running identity.

See the [Databricks Runtime 17.3 LTS ML release notes](https://docs.databricks.com/aws/en/release-notes/runtime/17.3lts-ml) and [Databricks local-file guidance](https://docs.databricks.com/aws/en/files/write-data) for current platform details.

The TPC-H dataset contains `customer`, `orders`, `lineitem`, and `part` with the columns used below. The notebook materializes local graph data and adds a synthetic primary key for `lineitem`.

## 1. Enter your configuration

Set the destination Unity Catalog catalog and schema for the views and output. The sample source remains `samples.tpch`. Kumo uses the configured local scratch directory to materialize graph and training data. A Unity Catalog Volume retains model bundles. Set the prediction date within the data's date range (1992-01-01 to 1998-08-02).


In [ ]:
from datetime import datetime
from pathlib import Path

# ── TPC-H source (Databricks sample data) ──
TPCH_CATALOG = "samples"
TPCH_SCHEMA = "tpch"

# ── Databricks connector ──
CONNECTOR_CATALOG = "<catalog>"                      # Your Unity Catalog catalog
CONNECTOR_SCHEMA = "<schema>"                        # Schema for TPC-H views and output
SQL_WAREHOUSE_ID = "<warehouse_id>"                  # SQL warehouse for the DatabricksConnector

# ── Local storage and model settings ──
# Classic GPU with dedicated access mode (default):
SCRATCH_DIR = "/local_disk0/tmp/kumoai-local"
# Serverless GPU: comment out the classic line above and uncomment this local temporary path.
# SCRATCH_DIR = "/tmp/kumoai-local"
ARTIFACT_ROOT = Path("/Volumes/<catalog>/<schema>/<volume>/tpch-runs")
MODEL_NAME = "TPCH_CUSTOMER_ORDER_MODEL"
PREDICTION_TIME = datetime(1998, 1, 1)              # Orders range 1992-01-01 to 1998-08-02.
PREDICTION_OUTPUT_TABLE = "TPCH_ORDER_PREDICTIONS"

# Durable prediction output.
WRITE_PREDICTIONS = True


## 2. Install Kumo Engine from PyPI

Run the installation cell once, then restart Python. After the restart, rerun configuration and continue at imports.


In [ ]:
%pip install "kumo-engine[databricks]"

dbutils.library.restartPython()


<a id="runtime"></a>
## Import and configure scratch

After each kernel restart, run configuration and these cells. They do not depend on variables created during installation. Configure scratch before constructing any Kumo table, graph, or trainer.

In [ ]:
from importlib.metadata import version
import shutil
import tempfile
from uuid import uuid4

import pandas as pd
from IPython.display import display

import kumo.engine as kumo
from kumoapi.typing import Dtype, Stype
from kumo.engine import Edge
from kumoapi.jobs import PredictionArtifactType, WriteMode


In [ ]:
# Create local scratch and retained-artifact directories, test write access, configure Kumo, and verify GPU access.
import sys
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"Use Python 3.12; found {sys.version.split()[0]}.")
scratch_root = Path(SCRATCH_DIR).expanduser().resolve()
if not (
    scratch_root.is_relative_to(Path("/local_disk0"))
    or scratch_root.is_relative_to(Path("/tmp"))
):
    raise ValueError("Use /local_disk0 for classic dedicated compute or /tmp for Serverless GPU.")
if not str(ARTIFACT_ROOT).startswith("/Volumes/") or "<" in str(ARTIFACT_ROOT):
    raise ValueError("Set ARTIFACT_ROOT to an existing writable Unity Catalog Volume path.")
if not ARTIFACT_ROOT.parent.is_dir():
    raise FileNotFoundError("Create the Unity Catalog Volume before running this notebook.")
scratch_root.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / "staging").mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryFile(dir=scratch_root) as probe:
    probe.write(b"kumo scratch check")
    probe.flush()
with tempfile.TemporaryFile(dir=ARTIFACT_ROOT) as probe:
    probe.write(b"artifact volume check")
    probe.flush()

kumo.configure(scratch_dir=str(scratch_root))
print("Python:", sys.version.split()[0])
print("Kumo version:", kumo.__version__)
print("GA package:", version("kumo-engine"))
print("Scratch:", scratch_root)
print("Retained artifacts:", ARTIFACT_ROOT)
print("Available scratch GiB:", round(shutil.disk_usage(scratch_root).free / 2**30, 1))

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Use a Databricks GPU cluster with a compatible runtime.")
for index in range(torch.cuda.device_count()):
    device = torch.cuda.get_device_properties(index)
    print("GPU:", device.name, "VRAM GiB:", round(device.total_memory / 2**30, 1))


The printed version identifies the installed build. The probe checks write access and free space at this path; it does not establish that the path is on supported local storage. Confirm the backing disk and capacity before materializing data.

## Connect to TPC-H data

The notebook creates Unity Catalog views of the four TPC-H sample tables in the configured schema. The `lineitem` view adds a synthetic single-column primary key (`lineitem_id`) since TPC-H uses a composite key. A `DatabricksConnector` uses the notebook's default auth chain — no separate credentials are needed.


In [ ]:
# ── Create TPC-H views in the connector schema ──
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}")

for table in ["customer", "orders", "part"]:
    spark.sql(f"""
        CREATE OR REPLACE VIEW {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.{table} AS
        SELECT * FROM {TPCH_CATALOG}.{TPCH_SCHEMA}.{table}
    """)
    print(f"View: {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.{table}")

# Lineitem: add synthetic single-column primary key (composite key not supported)
spark.sql(f"""
    CREATE OR REPLACE VIEW {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.lineitem AS
    SELECT *, CAST(l_orderkey * 8 + l_linenumber AS BIGINT) AS lineitem_id
    FROM {TPCH_CATALOG}.{TPCH_SCHEMA}.lineitem
""")
print(f"View: {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.lineitem (with lineitem_id)")

# ── Create Kumo Databricks connector (uses notebook's default auth) ──
connector = kumo.DatabricksConnector.from_notebook(
    catalog=CONNECTOR_CATALOG,
    schema_name=CONNECTOR_SCHEMA,
    warehouse_id=SQL_WAREHOUSE_ID,
)
print("Tables:", connector.table_names())


## Describe the four tables

A primary key identifies a row. A physical type describes its stored representation, such as a date; a semantic type describes its modeling role, such as a category or numerical value. `with_column()` returns an updated definition.

The customer and part definitions select the columns used in this example. TPC-H column names use the standard `c_`, `o_`, `l_`, `p_` prefixes.

In [ ]:
customer = (
    kumo.Table.from_source_table(
        connector["customer"],
        ["c_custkey", "c_nationkey", "c_mktsegment", "c_acctbal"],
        primary_key="c_custkey",
    )
    .with_column("c_nationkey", stype=Stype.categorical)
    .with_column("c_mktsegment", stype=Stype.categorical)
    .with_column("c_acctbal", stype=Stype.numerical)
)

part = (
    kumo.Table.from_source_table(
        connector["part"],
        ["p_partkey", "p_name", "p_brand", "p_type", "p_size",
         "p_container", "p_retailprice"],
        primary_key="p_partkey",
    )
    .with_column("p_name", stype=Stype.text)
    .with_column("p_brand", stype=Stype.categorical)
    .with_column("p_type", stype=Stype.categorical)
    .with_column("p_size", stype=Stype.numerical)
    .with_column("p_container", stype=Stype.categorical)
    .with_column("p_retailprice", stype=Stype.numerical)
)

Event timestamps let Kumo separate information available before a prediction boundary from future outcomes. Using future information as an input causes temporal leakage.

This sample uses `o_orderdate` for orders and `l_shipdate` for line items. The ship date represents when the item became observable in the fulfillment process and is a reasonable availability timestamp for this dataset.

In [ ]:
orders = (
    kumo.Table.from_source_table(
        connector["orders"],
        ["o_orderkey", "o_custkey", "o_orderdate", "o_orderstatus",
         "o_totalprice", "o_orderpriority", "o_shippriority"],
        primary_key="o_orderkey",
        time_column="o_orderdate",
    )
    .with_column("o_custkey", stype=Stype.ID)
    .with_column("o_orderdate", dtype=Dtype.date)
    .with_column("o_orderstatus", stype=Stype.categorical)
    .with_column("o_totalprice", stype=Stype.numerical)
    .with_column("o_orderpriority", stype=Stype.categorical)
    .with_column("o_shippriority", stype=Stype.numerical)
)

In [ ]:
lineitem = (
    kumo.Table.from_source_table(
        connector["lineitem"],
        ["lineitem_id", "l_orderkey", "l_partkey", "l_shipdate",
         "l_quantity", "l_extendedprice", "l_discount", "l_tax",
         "l_returnflag", "l_linestatus", "l_shipmode"],
        primary_key="lineitem_id",
        time_column="l_shipdate",
    )
    .with_column("l_orderkey", stype=Stype.ID)
    .with_column("l_partkey", stype=Stype.ID)
    .with_column("l_shipdate", dtype=Dtype.date)
    .with_column("l_quantity", stype=Stype.numerical)
    .with_column("l_extendedprice", stype=Stype.numerical)
    .with_column("l_discount", stype=Stype.numerical)
    .with_column("l_tax", stype=Stype.numerical)
    .with_column("l_returnflag", stype=Stype.categorical)
    .with_column("l_linestatus", stype=Stype.categorical)
    .with_column("l_shipmode", stype=Stype.categorical)
)

## Connect the tables and materialize a snapshot

A **graph** describes the tables and their relationships. Each arrow below goes from a foreign-key column to the destination table's primary key:

```text
lineitem.l_partkey   -> part.p_partkey
lineitem.l_orderkey  -> orders.o_orderkey
orders.o_custkey     -> customer.c_custkey
```

Validation checks the graph definition; it is not a blanket guarantee of source-data quality. A **snapshot** materializes source data into the local graph index used for training and prediction. Snapshot creation reads the tables and consumes local disk. Reuse requires an unchanged, valid retained snapshot; loading a model does not restore a lost index.

In [ ]:
graph = kumo.Graph(
    tables=[customer, part, orders, lineitem],
    edges=[
        Edge(src_table="orders", fkey="o_custkey", dst_table="customer"),
        Edge(src_table="lineitem", fkey="l_orderkey", dst_table="orders"),
        Edge(src_table="lineitem", fkey="l_partkey", dst_table="part"),
    ],
)

graph.validate()
print("graph id:", graph.id)

In [ ]:
snapshot = graph.snapshot()
print("Snapshot:", snapshot.id)
print("Local index:", snapshot.index_path)

## Define the prediction question

Predictive Query Language (PQL) is Kumo Engine's declarative syntax for defining a predictive modeling task: it specifies the target, the entity to score, and the data context. This default query predicts whether a customer who placed an order in the previous year will place another order in the next 90 days. The `WHERE` clause selects that active-customer population, and `> 0` makes the target binary.

In [ ]:
# Binary classification: predict whether a customer active in the past year will order again in the next 90 days.
pquery = kumo.PredictiveQuery(
    graph=graph,
    query="""
        PREDICT COUNT(orders.*, 0, 90, days) > 0
        FOR EACH customer.c_custkey
        WHERE COUNT(orders.*, -365, 0, days) > 0
    """,
)

pquery.validate()
print("Task type:", pquery.get_task_type())

### Optional task alternatives

The next two code cells are commented out. To try one, skip or comment out the binary query cell above, uncomment exactly one alternative, and run it before training. Both define `pquery` and use the existing `graph`, `snapshot`, and downstream variables. Train a new model for the selected task. The link example ranks parts associated with orders, not customer-level recommendations.

In [ ]:
# Regression: predict how many orders a customer active in the past year will place in the next 90 days.
# pquery = kumo.PredictiveQuery(
#     graph=graph,
#     query=(
#         "PREDICT COUNT(orders.*, 0, 90, days) "
#         "FOR EACH customer.c_custkey "
#         "WHERE COUNT(orders.*, -365, 0, days) > 0"
#     ),
# )
# pquery.validate()
# print("Task type:", pquery.get_task_type())

In [ ]:
# Link prediction: rank up to five parts that may ship in the next 90 days for an order with recent line items.
# pquery = kumo.PredictiveQuery(
#     graph=graph,
#     query=(
#         "PREDICT LIST_DISTINCT(lineitem.l_partkey, 0, 90, days) "
#         "RANK TOP 5 FOR EACH orders.o_orderkey "
#         "WHERE COUNT(lineitem.*, -365, 0, days) > 0"
#     ),
# )
# pquery.validate()
# print("Task type:", pquery.get_task_type())

### Generate a Training Table

After your query is validated, generate a training table to use for model fitting.

In [ ]:
train_table = pquery.generate_training_table(snapshot)
print("Training table:", train_table.train_table_path)


## Create a Trainer

The suggested model plan controls architecture, sampling, and optimization. The model plan is fully customizable: inspect and adjust its settings for your task, then create `trainer = kumo.Trainer(model_plan)` before fitting.

In [ ]:
model_plan = pquery.suggest_model_plan(snapshot)
model_plan.optimization.max_epochs = 2
# Create a Trainer with your model plan.
trainer = kumo.Trainer(model_plan)
print(model_plan)
print("Validation selection metric:", getattr(
    model_plan.training_job, "tune_metric", "not exposed by this build"))


### Train a Model

After creating a Trainer from the model plan, call `fit()` with the training table and compatible graph snapshot. This cell then copies the complete model bundle to the configured Unity Catalog Volume. Each run receives a unique retained path. Running the cell again starts another fit.


In [ ]:
# Serverless GPU only: if Optuna child processes hit IPC authentication errors,
# uncomment this block before fit(). Keep it commented on classic GPU compute.
# import multiprocessing
# try:
#     multiprocessing.set_start_method("spawn", force=True)
# except RuntimeError:
#     pass

training_result = trainer.fit(train_table, snapshot=snapshot)

run_root = ARTIFACT_ROOT / MODEL_NAME / uuid4().hex
run_root.mkdir(parents=True, exist_ok=False)
shutil.copytree(Path(training_result.bundle_dir), run_root / "model")
print("Retained model bundle:", run_root / "model")


### Inspect the selected result

Review the reported metrics and their stage names. Available metrics vary by task and build. The retained model bundle preserves model state; it does not archive the complete experiment history.


In [ ]:
print("Available metrics:", training_result.metrics)
print("Reusable local bundle:", training_result.bundle_dir)
for name, path in training_result.artifact_paths.items():
    print(name, path)


<a id="prediction"></a>
## Generate Batch Predictions

After training a model, generate batch predictions and write them to a Unity Catalog table. `WRITE_PREDICTIONS = True` enables that write by default; set it to `False` for local-only results.

Check the TPC-H order-date range against `PREDICTION_TIME` before scoring. A chosen date alone does not guarantee the following 90 days of observed outcomes for historical evaluation.

In [ ]:
date_range = spark.sql(f"""
    SELECT MIN(o_orderdate) AS first_order, MAX(o_orderdate) AS last_order
    FROM {TPCH_CATALOG}.{TPCH_SCHEMA}.orders
""")
display(date_range.toPandas())
print("Configured prediction boundary:", PREDICTION_TIME)


### Generate the scoring population

For these temporal queries, use a timezone-naive timestamp interpreted as UTC and reuse the fitted trainer's saved training-table generation plan. Binary classification and regression score customers with an order in the past year; the link example scores orders with recent line items. This timestamp example does not apply to static queries.

In [ ]:
prediction_table = pquery.generate_prediction_table(
    snapshot,
    prediction_time=PREDICTION_TIME,
    plan=trainer.training_table_generation_plan,
)

### Score locally, with an optional Unity Catalog output

**`WRITE_PREDICTIONS = True` authorizes `OVERWRITE`, which replaces the configured destination table's contents.** Choose a dedicated Unity Catalog table you own, never a source table. Set `WRITE_PREDICTIONS = False` to keep results only in local scratch.


In [ ]:
prediction_output = None
if WRITE_PREDICTIONS:
    import re
    from kumo_databricks.output import SEPARATOR as _SEP
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREDICTION_OUTPUT_TABLE):
        raise ValueError("Choose a simple dedicated output table name in configuration.")
    source_names = {"CUSTOMER", "PART", "ORDERS", "LINEITEM"}
    if PREDICTION_OUTPUT_TABLE.upper() in source_names:
        raise ValueError("Prediction output must not replace a source table.")
    prediction_output = kumo.PredictionOutputConfig(
        connector=connector, artifact_type=PredictionArtifactType.PREDICTIONS,
        table_name=f"{CONNECTOR_SCHEMA}{_SEP}{PREDICTION_OUTPUT_TABLE}",
        write_mode=WriteMode.OVERWRITE,
    )
from kumoapi.task import TaskType

prediction_options = {}
if pquery.get_task_type() == TaskType.BINARY_CLASSIFICATION:
    prediction_options["binary_classification_threshold"] = 0.5
elif pquery.get_task_type() == TaskType.TEMPORAL_LINK_PREDICTION:
    prediction_options["num_classes_to_return"] = 5
prediction_result = trainer.predict(
    prediction_table, snapshot=snapshot,
    output_config=prediction_output, **prediction_options,
)
print("Rows:", prediction_result.num_rows)
print("Local Parquet files:", prediction_result.files)
if WRITE_PREDICTIONS:
    print("Output:",
          f"{CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.{PREDICTION_OUTPUT_TABLE}")


### Read a bounded preview

Read at most ten rows across the local Parquet files in batches. This avoids collecting the entire result. Parquet readers still need file metadata and decoding buffers. `prediction_result.frame` instead loads all rows into memory, even if followed by `.head()`.

In [ ]:
import pyarrow.parquet as pq

def preview_parquet(files, limit=10):
    frames = []
    remaining = limit
    for path in files:
        if remaining <= 0:
            break
        with pq.ParquetFile(path) as parquet:
            for batch in parquet.iter_batches(batch_size=remaining):
                frames.append(batch.slice(0, remaining).to_pandas())
                remaining -= len(frames[-1])
                if remaining <= 0:
                    break
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

display(preview_parquet(prediction_result.files))